In [0]:
%run ./01-config

In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import floor, months_between, current_date, when, col


# --- Configuration ---
spark.sql(f"USE {catalog}.{db_name}")
start = int(time.time())
print("Starting silver layer batch upserts...")

# ==============================================================================
# SILVER LAYER STAGE 1: Core Entity Tables
# ==============================================================================

# --- 1. Users (from registered_users_bz) ---
print("Upserting users...", end='')
df_users_src = (spark.read.table(f"{catalog}.{db_name}.registered_users_bz")
    .selectExpr("user_id", "device_id", "mac_address",
                "cast(registration_timestamp as timestamp) as registration_timestamp")
    .dropDuplicates(["user_id", "device_id"])
)
df_users_src.createOrReplaceTempView("users_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.users a
    USING users_delta b
    ON a.user_id = b.user_id
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

# --- 2. Gym Logs (from gym_logins_bz) ---
print("Upserting gym_logs...", end='')
df_gym_src = (spark.read.table(f"{catalog}.{db_name}.gym_logins_bz")
    .selectExpr("mac_address", "gym",
                "cast(login as timestamp) as login",
                "cast(logout as timestamp) as logout")
    .dropDuplicates(["mac_address", "gym", "login"])
)
df_gym_src.createOrReplaceTempView("gym_logs_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.gym_logs a
    USING gym_logs_delta b
    ON a.mac_address = b.mac_address AND a.gym = b.gym AND a.login = b.login
    WHEN MATCHED AND b.logout > a.login AND b.logout > a.logout
        THEN UPDATE SET logout = b.logout
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

# --- 3. User Profile (CDC from kafka_multiplex_bz, topic='user_info') ---
print("Upserting user_profile...", end='')
schema_profile = """
    user_id bigint, update_type STRING, timestamp FLOAT,
    dob STRING, sex STRING, gender STRING, first_name STRING, last_name STRING,
    address STRUCT<street_address: STRING, city: STRING, state: STRING, zip: INT>
"""

window_profile = Window.partitionBy("user_id").orderBy(F.col("updated").desc())

df_profile_src = (spark.read.table(f"{catalog}.{db_name}.kafka_multiplex_bz")
    .filter("topic = 'user_info'")
    .select(F.from_json(F.col("value").cast("string"), schema_profile).alias("v"))
    .select("v.*")
    .select(
        "user_id",
        F.to_date("dob", "MM/dd/yyyy").alias("dob"),
        "sex", "gender", "first_name", "last_name",
        "address.*",
        F.col("timestamp").cast("timestamp").alias("updated"),
        "update_type"
    )
    .filter(F.col("update_type").isin(["new", "update"]))
    .withColumn("rank", F.row_number().over(window_profile))
    .filter("rank == 1")
    .drop("rank", "update_type")
)
df_profile_src.createOrReplaceTempView("user_profile_cdc")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.user_profile a
    USING user_profile_cdc b
    ON a.user_id = b.user_id
    WHEN MATCHED AND a.updated < b.updated
        THEN UPDATE SET *
    WHEN NOT MATCHED
        THEN INSERT *
""")
print("Done")

# --- 4. Workouts (from kafka_multiplex_bz, topic='workout') ---
print("Upserting workouts...", end='')
schema_workout = "user_id INT, workout_id INT, timestamp FLOAT, action STRING, session_id INT"

df_workouts_src = (spark.read.table(f"{catalog}.{db_name}.kafka_multiplex_bz")
    .filter("topic = 'workout'")
    .select(F.from_json(F.col("value").cast("string"), schema_workout).alias("v"))
    .select("v.*")
    .select("user_id", "workout_id",
            F.col("timestamp").cast("timestamp").alias("time"),
            "action", "session_id")
    .dropDuplicates(["user_id", "time"])
)
df_workouts_src.createOrReplaceTempView("workouts_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.workouts a
    USING workouts_delta b
    ON a.user_id = b.user_id AND a.time = b.time
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

# --- 5. Heart Rate (from kafka_multiplex_bz, topic='bpm') ---
print("Upserting heart_rate...", end='')
schema_bpm = "device_id LONG, time TIMESTAMP, heartrate DOUBLE"

df_hr_src = (spark.read.table(f"{catalog}.{db_name}.kafka_multiplex_bz")
    .filter("topic = 'bpm'")
    .select(F.from_json(F.col("value").cast("string"), schema_bpm).alias("v"))
    .select("v.*", F.when(F.col("v.heartrate") <= 0, False).otherwise(True).alias("valid"))
    .dropDuplicates(["device_id", "time"])
)
df_hr_src.createOrReplaceTempView("heart_rate_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.heart_rate a
    USING heart_rate_delta b
    ON a.device_id = b.device_id AND a.time = b.time
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

print(f"\nSilver layer Stage 1 completed in {int(time.time()) - start} seconds")

# ==============================================================================
# SILVER LAYER STAGE 2: Derived Tables
# ==============================================================================

# --- 6. User Bins (demographic segmentation from user_profile + users) ---
print("Upserting user_bins...", end='')

# Age binning logic inline
age_col = floor(months_between(current_date(), col("dob")) / 12)
age_bin_expr = (when(age_col < 18, "under 18")
    .when((age_col >= 18) & (age_col < 25), "18-25")
    .when((age_col >= 25) & (age_col < 35), "25-35")
    .when((age_col >= 35) & (age_col < 45), "35-45")
    .when((age_col >= 45) & (age_col < 55), "45-55")
    .when((age_col >= 55) & (age_col < 65), "55-65")
    .when((age_col >= 65) & (age_col < 75), "65-75")
    .when((age_col >= 75) & (age_col < 85), "75-85")
    .when((age_col >= 85) & (age_col < 95), "85-95")
    .when(age_col >= 95, "95+")
    .otherwise("invalid age"))

df_user_ids = spark.table(f"{catalog}.{db_name}.users").select("user_id").distinct()

df_bins_src = (spark.read.table(f"{catalog}.{db_name}.user_profile")
    .dropDuplicates(["user_id"])
    .join(df_user_ids, on="user_id", how="left")
    .select(
        "user_id",
        age_bin_expr.alias("age"),
        "gender", "city", "state"
    )
)
df_bins_src.createOrReplaceTempView("user_bins_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.user_bins a
    USING user_bins_delta b
    ON a.user_id = b.user_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

# --- 7. Completed Workouts (join start + stop events) ---
print("Upserting completed_workouts...", end='')

df_start = (spark.read.table(f"{catalog}.{db_name}.workouts")
    .filter("action = 'start'")
    .selectExpr("user_id", "workout_id", "session_id", "time as start_time")
)

df_stop = (spark.read.table(f"{catalog}.{db_name}.workouts")
    .filter("action = 'stop'")
    .selectExpr("user_id as stop_user_id", "workout_id as stop_workout_id",
                "session_id as stop_session_id", "time as end_time")
)

df_completed_src = (df_start.join(df_stop,
    (df_start.user_id == df_stop.stop_user_id) &
    (df_start.workout_id == df_stop.stop_workout_id) &
    (df_start.session_id == df_stop.stop_session_id) &
    (df_stop.end_time < df_start.start_time + F.expr('interval 3 hours')))
    .select("user_id", "workout_id", "session_id", "start_time", "end_time")
)
df_completed_src.createOrReplaceTempView("completed_workouts_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.completed_workouts a
    USING completed_workouts_delta b
    ON a.user_id = b.user_id AND a.workout_id = b.workout_id AND a.session_id = b.session_id
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

print(f"Silver layer Stage 2 completed in {int(time.time()) - start} seconds")

# ==============================================================================
# SILVER LAYER STAGE 3: Session-Level Aggregations
# ==============================================================================

# --- 8. Workout BPM (link heart rate to completed workouts) ---
print("Upserting workout_bpm...", end='')

df_users_map = spark.read.table(f"{catalog}.{db_name}.users")

df_cw = (spark.read.table(f"{catalog}.{db_name}.completed_workouts")
    .join(df_users_map, "user_id")
    .selectExpr("user_id", "device_id", "workout_id", "session_id", "start_time", "end_time")
)

df_bpm = (spark.read.table(f"{catalog}.{db_name}.heart_rate")
    .filter("valid = True")
    .selectExpr("device_id as bpm_device_id", "time", "heartrate")
)

df_workout_bpm_src = (df_bpm.join(df_cw,
    (df_cw.device_id == df_bpm.bpm_device_id) &
    (df_bpm.time > df_cw.start_time) &
    (df_bpm.time <= df_cw.end_time))
    .select("user_id", "workout_id", "session_id", "start_time", "end_time", "time", "heartrate")
)
df_workout_bpm_src.createOrReplaceTempView("workout_bpm_delta")
spark.sql(f"""
    MERGE INTO {catalog}.{db_name}.workout_bpm a
    USING workout_bpm_delta b
    ON a.user_id = b.user_id AND a.workout_id = b.workout_id
       AND a.session_id = b.session_id AND a.time = b.time
    WHEN NOT MATCHED THEN INSERT *
""")
print("Done")

print(f"\nSilver layer batch upserts completed in {int(time.time()) - start} seconds")

In [0]:
# ==============================================================================
# SILVER LAYER VALIDATION (Batch)
# Set `sets` to 1 for a single batch load, 2 for two batches
# ==============================================================================
import time

sets = 1  # <-- UPDATE: 1 for single batch, 2 for double batch
start_val = int(time.time())
print("Starting Silver layer validation...\n")

# --- Silver Layer 1: Core Entity Tables ---
validations = [
    ("users", 5 if sets == 1 else 10, "true"),
    ("gym_logs", 8 if sets == 1 else 16, "true"),
    ("user_profile", 5 if sets == 1 else 10, "true"),
    ("workouts", 16 if sets == 1 else 32, "true"),
    ("heart_rate", sets * 253801, "true"),
]

for table_name, expected_count, filter_expr in validations:
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter_expr).count()
    assert actual_count == expected_count, \
        f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}"
    print(f"  {table_name}: {actual_count:,} / {expected_count:,} records - OK")

print(f"\nSilver layer 1 validation done in {int(time.time()) - start_val} seconds\n")

# --- Silver Layer 2: Derived Tables ---
validations_2 = [
    ("user_bins", 5 if sets == 1 else 10, "true"),
    ("completed_workouts", 8 if sets == 1 else 16, "true"),
]

for table_name, expected_count, filter_expr in validations_2:
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter_expr).count()
    assert actual_count == expected_count, \
        f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}"
    print(f"  {table_name}: {actual_count:,} / {expected_count:,} records - OK")

print(f"\nSilver layer 2 validation done in {int(time.time()) - start_val} seconds\n")

# --- Silver Layer 3: Session-Level Aggregations ---
table_name = "workout_bpm"
expected_count = 3968 if sets == 1 else 8192
actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").count()
assert actual_count == expected_count, \
    f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}"
print(f"  {table_name}: {actual_count:,} / {expected_count:,} records - OK")

print(f"\nSilver layer validation completed in {int(time.time()) - start_val} seconds")